In [1]:
from jqdata import *
from jqlib.technical_analysis import *
from jqfactor import get_factor_values
from jqfactor import winsorize_med
from jqfactor import standardlize
from jqfactor import neutralize
import datetime
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels import regression
from six import StringIO
#导入pca
from sklearn.decomposition import PCA
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.grid_search import GridSearchCV
from sklearn import metrics
from tqdm import tqdm
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
#获取指定周期的日期列表 'W、M、Q'
def get_period_date(peroid,start_date, end_date):
    #设定转换周期period_type  转换为周是'W',月'M',季度线'Q',五分钟'5min',12天'12D'
    stock_data = get_price('000001.XSHE',start_date,end_date,'daily',fields=['close'])
    stock_data['date']=stock_data.index
    period_stock_data=stock_data.resample(peroid,how='last')
    period_stock_data = period_stock_data.set_index('date').dropna()
    date=period_stock_data.index
    pydate_array = date.to_pydatetime()
    date_only_array = np.vectorize(lambda s: s.strftime('%Y-%m-%d'))(pydate_array )
    date_only_series = pd.Series(date_only_array)
    start_date = datetime.datetime.strptime(start_date, "%Y-%m-%d")
    start_date=start_date-datetime.timedelta(days=1)
    start_date = start_date.strftime("%Y-%m-%d")
    date_list=date_only_series.values.tolist()
    date_list.insert(0,start_date)
    return date_list
peroid = 'W'
# start_date = '2009-01-01'
# end_date = '2025-05-29'
# DAY = get_period_date(peroid,start_date, end_date)
# print(len(DAY))

/opt/conda/lib/python3.6/site-packages/sklearn/cross_validation.py:41: DeprecationWarning: This module was deprecated in version 0.18 in favor of the model_selection module into which all the refactored classes and functions are moved. Also note that the interface of the new CV iterators are different from that of this module. This module will be removed in 0.20.
  "This module will be removed in 0.20.", DeprecationWarning)
/opt/conda/lib/python3.6/site-packages/sklearn/grid_search.py:42: DeprecationWarning: This module was deprecated in version 0.18 in favor of the model_selection module into which all the refactored classes and functions are moved. This module will be removed in 0.20.
  DeprecationWarning)


In [2]:
# 你的因子列表和对应系数
# factor_list = [
#     (['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60'], [-0.0010960853, -0.0000019512, 0.0001972877, 0.0297629353], -0.0296314300),
#     (['natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26'], [-0.0018352322, 0.0002960791, -0.0006454540, 0.0590388494], -0.0154238607),
#     (['CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26'], [-0.0000056065, -0.0016725631, 0.0066194434, 0.0452138428], -0.0145511631),
#     (['natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio'], [-0.0016593531, -0.0179428833, 0.0777040441, -0.0000220915], -0.0221400742),
#     (['BIAS60', 'CR20', 'PLRC6', 'Variance120'], [-0.0002890613, 0.0000329763, -0.1972336257, 0.0000102071], -0.0028990305),
#     (['VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down'], [-0.0000006067, 0.0010360723, -0.0394544384, 0.0169230822, -0.0203055230], 0.0012359940),
#     (['debt_to_assets', 'BIAS10', 'ROC120', 'leverage'], [0.0037088213, -0.0006521093, -0.0000285698, -0.0015515131], -0.0018919777),
#     (['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [0.0002558345, -0.0023334987, -0.0023388390], 0.0510282617),
#     (['liquidity', 'roa_ttm'], [-0.0027488974, -0.0034165195], 0.0014585062),
#     (['VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle'], [-0.0000000005, -0.0000005805, -0.0006148054, 0.0000004323], 0.0014091609),
#     (['operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income'], [0.0000018895, -0.0001319373, -0.0000000005, -0.0000002337], 0.0015024743),
#     (['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'], [0.0000107905, -0.0003682269, 0.0127048800], 0.0005662943),
#     (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-0.0000554025, 0.0000998803, -0.0000000001], 0.0014947396),
#     (['non_current_asset_ratio', 'admin_expense_rate', 'VOL20'], [-0.0006165800, 0.0003081868, -0.0006966830], 0.0027891700),
#     (['MAC5', 'MLEV', 'EMAC120', 'fifty_two_week_close_rank'], [0.0851601473, 0.0013219219, 0.0364972371, -0.0000355833], -0.1173515269),
#     (['turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12'], [-0.1337048931, -0.0019793370, 0.0000005699, 0.0814894653], -0.0783076150),
#     (['MACDC', 'CCI20', 'EMAC20'], [0.1388054196, 0.0000138476, 0.1224230943], -0.1223598577),
#     (['single_day_VPT_6', 'EMAC120'], [-0.0000000969, 0.0252008767], -0.0249596372),
#     (['Volume1M', 'BIAS60', 'Price1M', 'cash_earnings_to_price_ratio'], [0.0552286754, -0.0001924418, -0.0252638402, 0.0030728082], 0.0005356183),
#     (['Variance120', 'EMAC26', 'VOL120', 'bull_power', 'EMA5'], [-0.0000100918, 0.0648095476, -0.0004998660, 0.0314640191, 0.0472963282], -0.1110559732),
#     (['BIAS10', 'TRIX10', 'BIAS5'], [0.0003488878, -0.0050460763, -0.0015468061], 0.0005696761),  
#   ]


#################
# factor_list = [
#     (['cash_flow_to_price_ratio', 'sales_to_price_ratio', 'VOL10', 'DAVOL20', 'ARBR', 'VROC6', 'VSTD20', 'VR', 'VDIFF', 'AR', 'VEMA5', 'VROC12'], [-1.117e-05, -1.06919e-05, -1.68409e-05, 3.8552e-06, -1.854e-07, -6.048e-07, 4e-10, -6.6461e-06, -6e-10, -5.8533e-06, -3e-10, 2.75e-08], 0.0025990645),
#     (['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [0.0002558345, -0.0023334987, -0.002338839], 0.0510282617),
#     (['liquidity', 'roa_ttm'], [-0.0027488974, -0.0034165195], 0.0014585062),
#     (['VEMA12', 'VSTD10', 'single_day_VPT_6'], [1e-10, -6e-10, -2.126e-07], 0.0018854583),
#     (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-5.54025e-05, 9.98803e-05, -1e-10], 0.0014947396),

               
#     (['momentum', 'current_ratio', 'Variance20', 'equity_to_asset_ratio'], [0.0005177503, 0.0000188540, -0.0000233985, 0.0002239414], 0.0021646292),
#     (['accounts_payable_turnover_rate', 'net_operating_cash_flow_coverage', 'quick_ratio'], [0.0000432407, 0.0000001756, 0.0000176207], 0.0023525561),
#     (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-0.0000507037, -0.0000781977, 0.0000000194], 0.0019693262),
#     (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [0.0000000194, -0.0005124841, -0.0011852668, -0.0030929328], 0.0704695322),
#     (['roic_ttm', 'net_operate_cash_flow_per_share', 'EMAC120'], [-0.0012191355, 0.0011824701, 0.0155764421], -0.0149666230),
#     (['MFI14', 'DEGM_8y', 'MAC10', 'EMAC26'], [0.0000763204, -0.0029489569, -0.0420481459, 0.0708067423], -0.0324936447),
    
#     (['ARBR', 'SGAI', 'net_profit_to_total_operate_revenue_ttm', 'retained_profit_per_share'], [-0.0000007909, 0.0010129565, -0.0001096506, 0.0000277437], 0.0000750425),
#     (['Price1Y', 'total_profit_to_cost_ratio', 'VOL120'], [-0.0025236307, -0.0002478142, -0.0000478038], 0.0012926181),
#     (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-0.0000507163, -0.0000782166, 0.0000000194], 0.0019697869),
#     (['debt_to_assets', 'operating_cost_to_operating_revenue_ratio', 'DAVOL20', 'price_no_fq', 'sales_growth'], [-0.0024950414, 0.0016242826, -0.0017688675, -0.0000503276, 0.0013920485], 0.0036558758),
#     (['cashflow_per_share_ttm', 'sharpe_ratio_120'], [0.0000315416, -0.0003025576], 0.0021910628),

# ]


###最近的参数，分层显著性最高




###################历史的参数，R2稳定性最好
###['2025-05-20', '2025-06-07'] 滚动验证
factor_list = [
    (['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [0.0002666821, -0.0020518674, -0.0023101097], 0.0507803593),
    (['size', 'roe_ttm', 'current_asset_turnover_rate'], [-0.0008979094, -0.0000039691, 0.0002272270], -0.0003337466),
    (['VOL10', 'single_day_VPT_12'], [-0.0006370810, -0.0000001720], 0.0027864796),
    (['adjusted_profit_to_total_profit'], [-0.0000013402], 0.0013302010),
    (['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'], [0.0000357266, -0.0003667557, 0.0130488065], 0.0002890622),
    (['cash_to_current_liability', 'operating_tax_to_operating_revenue_ratio_ttm', 'Price3M'], [-0.0003459985, 0.0010498108, -0.0233277951], 0.0013685457),
    (['liquidity', 'roa_ttm'], [-0.0027426855, -0.0027239563], 0.0013502358),
    (['VSTD10', 'ROC60'], [-0.0000000004, -0.0000823648], 0.0013880176),
]

##################
# factor_list = [
#     (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [-1e-10, 0.0002558345, -0.0023334987, -0.002338839], 0.05),
#     (['liquidity', 'roa_ttm'], [-0.0027488974, -0.0034165195], 0.0),
#     (['size', 'long_term_predicted_earnings_growth', 'earnings_to_price_ratio', 'earnings_yield', 'cash_earnings_to_price_ratio'], [-0.001884059, 0.0020108261, 0.0031710681, -0.0001211369, 0.0044158193], -0.0),
#     (['boll_up', 'turnover_volatility', 'size', 'sale_expense_ttm', 'momentum'], [0.0347870304, -0.2019477089, -0.0023226377, 0.0, 3.05239e-05], -0.04),
#     (['price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate'], [-5.54025e-05, 9.98803e-05, -1e-10], 0.0),
# ]
# factor_list = [
#     (['inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'], [0.0000000194, -0.0005124017, -0.0011854417, -0.0030933629], 0.0704793466),
#     (['size', 'roe_ttm', 'current_asset_turnover_rate'], [-0.0019726951, -0.0000182802, 0.0003513364], -0.0016236166),
#     (['VOL10', 'single_day_VPT_12'], [-0.0004570590, -0.0000001234], 0.0033578804),
#     (['adjusted_profit_to_total_profit'], [-0.0000013724], 0.0025696068),
#     (['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'], [0.0000746320, -0.0005792696, 0.0218890582], 0.0010033916),
#     (['cash_to_current_liability', 'operating_tax_to_operating_revenue_ratio_ttm', 'Price3M'], [-0.0005075893, 0.0015746797, -0.0106494228], 0.0024684325),
#     (['liquidity', 'roa_ttm'], [-0.0028631376, -0.0010335390], 0.0016439413),
# ]
##################历史的全量数据最全

  

# 获取所有清理后的因子名
# Extract first column of factor names
factor_names = [item[0] for item in factor_list]

for sublist in factor_names:
    print(f"xx.append({sublist})")




    
# 获取所有清理后的因子名
all_factors = []
for feature_names, _, _ in factor_list:
    all_factors.extend(feature_names)

# 打印所有因子名
print(all_factors)


# from collections import Counter

# # 获取所有清理后的因子名
# all_factors = []
# for feature_names, _, _ in cleaned_data:
#     all_factors.extend(feature_names)

# # 统计因子名出现的次数
# factor_counts = Counter(all_factors)

# # 打印因子出现次数
# for factor, count in factor_counts.items():
#     print(f"{factor}: {count}")

    

xx.append(['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap'])
xx.append(['size', 'roe_ttm', 'current_asset_turnover_rate'])
xx.append(['VOL10', 'single_day_VPT_12'])
xx.append(['adjusted_profit_to_total_profit'])
xx.append(['super_quick_ratio', 'cube_of_size', 'cfo_to_ev'])
xx.append(['cash_to_current_liability', 'operating_tax_to_operating_revenue_ratio_ttm', 'Price3M'])
xx.append(['liquidity', 'roa_ttm'])
xx.append(['VSTD10', 'ROC60'])
['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap', 'size', 'roe_ttm', 'current_asset_turnover_rate', 'VOL10', 'single_day_VPT_12', 'adjusted_profit_to_total_profit', 'super_quick_ratio', 'cube_of_size', 'cfo_to_ev', 'cash_to_current_liability', 'operating_tax_to_operating_revenue_ratio_ttm', 'Price3M', 'liquidity', 'roa_ttm', 'VSTD10', 'ROC60']


In [3]:



jqfactors_list = all_factors#['BIAS5', 'np_parent_company_owners_growth_rate', 'ROC6', 'MAC60', 'natural_log_of_market_cap', 'average_share_turnover_annual', 'VOL240', 'EMAC26', 'CCI10', 'natural_log_of_market_cap', 'boll_up', 'EMAC26', 'natural_log_of_market_cap', 'EMAC10', 'EMAC20', 'net_profit_ratio', 'BIAS60', 'CR20', 'PLRC6', 'Variance120', 'VROC6', 'equity_to_asset_ratio', 'Price1M', 'MAC120', 'boll_down', 'debt_to_assets', 'BIAS10', 'ROC120', 'leverage', 'inventory_turnover_rate', 'sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap', 'liquidity', 'roa_ttm', 'VSTD20', 'account_receivable_turnover_rate', 'long_term_debt_to_asset_ratio', 'OperatingCycle', 'operating_revenue_growth_rate', 'surplus_reserve_fund_per_share', 'VSTD20', 'net_operate_cash_flow_to_operate_income', 'super_quick_ratio', 'cube_of_size', 'cfo_to_ev', 'price_no_fq', 'total_profit_to_cost_ratio', 'inventory_turnover_rate', 'non_current_asset_ratio', 'admin_expense_rate', 'VOL20', 'MAC5', 'MLEV', 'EMAC120', 'fifty_two_week_close_rank', 'turnover_volatility', 'maximum_margin', 'equity_turnover_rate', 'EMAC12', 'TVSTD20', 'VOL5', 'growth', 'net_asset_growth_rate', 'EMA5', 'VOL60', 'TVSTD20', 'BBIC', 'goods_service_cash_to_operating_revenue_ttm']
# print(jqfactors_list)


In [4]:
#去除上市距beginDate不足3个月的股票
def delect_stop(stocks,beginDate,n=30*3):
    stockList=[]
    beginDate = datetime.datetime.strptime(beginDate, "%Y-%m-%d")
    for stock in stocks:
        start_date=get_security_info(stock).start_date
        if start_date<(beginDate-datetime.timedelta(days=n)).date():
            stockList.append(stock)
    return stockList
#获取股票池
def get_stock(stockPool,begin_date):
    if stockPool=='HS300':
        stockList=get_index_stocks('000300.XSHG',begin_date)
    elif stockPool=='ZZ500':
        stockList=get_index_stocks('399905.XSHE',begin_date)
    elif stockPool=='ZZ800':
        stockList=get_index_stocks('399906.XSHE',begin_date)   
    elif stockPool=='CYBZ':
        stockList=get_index_stocks('399006.XSHE',begin_date)
    elif stockPool=='ZXBZ':
        stockList=get_index_stocks('399005.XSHE',begin_date)
    elif stockPool=='A':
        stockList=get_index_stocks('000002.XSHG',begin_date)+get_index_stocks('399107.XSHE',begin_date)
        stockList = [stock for stock in stockList if not stock.startswith(('68', '4', '8'))]
    elif stockPool=='AA':
        stockList=get_index_stocks('000985.XSHG',begin_date)
        stockList = [stock for stock in stockList if not stock.startswith(('3', '68', '4', '8'))]
    elif stockPool=='small':
        stockList=get_index_stocks('399101.XSHE',begin_date)
        stockList = [stock for stock in stockList if not stock.startswith(('68', '4', '8'))]
    elif stockPool=='small_25':
        initial_list=get_index_stocks('000002.XSHG',begin_date)+get_index_stocks('399107.XSHE',begin_date)
        stockList = list(get_fundamentals(
        query(valuation.code,valuation.market_cap).filter(
            valuation.code.in_(initial_list),
            valuation.market_cap < 25
        ).order_by(
            valuation.circulating_market_cap.asc()
        )).code)[:50]
    elif stockPool=='small_fengzhi':
        initial_list = get_all_securities('stock', begin_date).index.tolist()
        stockList = list(get_fundamentals(
        query(valuation.code,valuation.market_cap).filter(
            valuation.code.in_(initial_list),
            valuation.market_cap < 80
        ).order_by(
            valuation.circulating_market_cap.asc()
        )).code)
        stockList = [stock for stock in stockList if not stock.startswith(('3','68', '4', '8'))]
    #剔除ST股
    st_data=get_extras('is_st',stockList, count = 1,end_date=begin_date)
    stockList = [stock for stock in stockList if not st_data[stock][0]]
    #剔除停牌、新股及退市股票
    stockList=delect_stop(stockList,begin_date)
    return stockList
#获取时间为date的全部因子数据
def get_factor_data(securities_list,date,jqfactors_list):
    print("###########",date)
    factor_data = get_factor_values(securities=securities_list, \
                                    factors=jqfactors_list, \
                                    count=1, \
                                    end_date=date)
    df_jq_factor=pd.DataFrame(index=securities_list)
    for i in factor_data.keys():
        df_jq_factor[i]=factor_data[i].iloc[0,:]
    return df_jq_factor
# dateList = get_period_date(peroid,start_date, end_date)
# print(len(dateList))

In [5]:


testday_sd = ['2025-09-01', '2025-09-10']

# 昨天，遇周末直接跳到上周五
y = datetime.datetime.now() - datetime.timedelta(days=1)
y -= datetime.timedelta(days=max(0, y.weekday() - 4))

testday_sd[1] = y.strftime('%Y-%m-%d')
print(testday_sd)


peroid = 'D'
start_date = testday_sd[0]
end_date = testday_sd[-1]
testday = get_period_date(peroid,start_date, end_date)
print(testday)

test_filename = f"testdata_{testday[0].replace('-', '')}_{testday[-1].replace('-', '')}.csv"

test_data=pd.DataFrame()
for date in tqdm(testday[:-1]):
    print(date)
    stockList = get_stock('small_fengzhi', date)
    print("stocknum:",len(stockList))
#     stockList = get_all_securities('stock', date).index.tolist()
#     print(f"初始股票数量: {len(stockList)}")
    factor_solve_data = get_factor_data(stockList, date, jqfactors_list)
    
    # 获取连续两个交易日的收盘价
    data_close = get_price(stockList, start_date=date, end_date=testday[testday.index(date)+1], frequency='1d', fields='close')['close']

#     print(testday[testday.index(date)+1])
#     print(data_close)
#     print(data_close.shape)

    # 如果拿到的是两天数据，才计算
    if data_close.shape[0] >= 2:
        factor_solve_data['pchg'] = data_close.iloc[-1] / data_close.iloc[0] - 1
    else:
        factor_solve_data['pchg'] = 0 #fengzhi 这里为了防止后续出错改成0

    factor_solve_data['date'] = date
    test_data = pd.concat([test_data, factor_solve_data], ignore_index=False)


        
test_data.to_csv(test_filename, index=True)
print(len(test_data))

['2025-09-01', '2025-11-10']


  0%|          | 0/45 [00:00<?, ?it/s]

['2025-08-31', '2025-09-01', '2025-09-02', '2025-09-03', '2025-09-04', '2025-09-05', '2025-09-08', '2025-09-09', '2025-09-10', '2025-09-11', '2025-09-12', '2025-09-15', '2025-09-16', '2025-09-17', '2025-09-18', '2025-09-19', '2025-09-22', '2025-09-23', '2025-09-24', '2025-09-25', '2025-09-26', '2025-09-29', '2025-09-30', '2025-10-09', '2025-10-10', '2025-10-13', '2025-10-14', '2025-10-15', '2025-10-16', '2025-10-17', '2025-10-20', '2025-10-21', '2025-10-22', '2025-10-23', '2025-10-24', '2025-10-27', '2025-10-28', '2025-10-29', '2025-10-30', '2025-10-31', '2025-11-03', '2025-11-04', '2025-11-05', '2025-11-06', '2025-11-07', '2025-11-10']
2025-08-31
stocknum: 1475
########### 2025-08-31


  2%|▏         | 1/45 [00:02<02:04,  2.84s/it]

2025-09-01
stocknum: 1475
########### 2025-09-01


...

2025-11-06
stocknum: 1473
########### 2025-11-06


 98%|█████████▊| 44/45 [01:35<00:02,  2.23s/it]

2025-11-07
stocknum: 1473
########### 2025-11-07


100%|██████████| 45/45 [01:38<00:00,  2.37s/it]


66380


In [6]:
test_data

,size,Price3M,operating_tax_to_operating_revenue_ratio_ttm,cube_of_size,single_day_VPT_12,share_turnover_monthly,roa_ttm,sales_to_price_ratio,current_asset_turnover_rate,VOL10,cfo_to_ev,super_quick_ratio,natural_log_of_market_cap,ROC60,VSTD10,liquidity,adjusted_profit_to_total_profit,cash_to_current_liability,roe_ttm,pchg,date
001260.XSHE,-2.965263,0.036047,0.011657,-7.395735,-115.103637,0.204311,0.044777,0.255840,1.144526,6.66996,0.025079,1.381418,21.561163,10.439560,9.858698e+05,0.614209,0.704537,0.234303,0.057959,0.000000,2025-08-31
603307.XSHG,-2.821670,0.022420,0.008910,-4.874145,-14.260510,0.019891,0.079353,0.318745,0.693764,4.64062,0.049538,4.885412,21.704179,7.614352,2.905355e+05,0.473580,0.684170,1.381531,0.093865,0.000000,2025-08-31
001366.XSHE,-3.115211,-0.018418,0.002723,-10.420746,-222.531241,0.189310,-0.034827,0.586786,2.169833,6.57253,-0.031297,1.304527,21.411817,2.481390,1.118411e+06,0.516061,0.797835,0.948845,-0.049266,0.000000,2025-08-31
001368.XSHE,-2.754425,0.033998,0.007900,-3.814387,4.721002,0.734093,0.055213,0.367809,1.111380,10.07489,0.047162,2.550607,21.771153,8.535529,1.189244e+06,1.323219,0.816856,0.903026,0.067935,0.000000,2025-08-31
605069.XSHG,-2.992182,-0.024486,0.005858,-7.908748,-456.712649,0.691165,-0.036297,0.160570,0.263441,8.40491,0.012614,0.473866,21.534352,-4.320432,1.436354e+06,1.377089,0.918650,0.005839,-0.123943,0.000000,2025-08-31
001335.XSHE,-2.439943,0.084500,NaN,0.179471,250.167553,0.763911,0.050470,0.371568,NaN,12.74372,NaN,0.919535,22.084371,13.286142,1.359019e+06,1.619178,0.911429,NaN,0.114896,0.000000,2025-08-31
603120.XSHG,-2.456771,0.036184,NaN,0.004080,125.643835,0.875751,0.059431,0.160251,NaN,13.67716,NaN,6.467708,22.067610,9.666581,7.771460e+05,1.980887,0.711655,NaN,0.067496,0.000000,2025-08-31
001367.XSHE,-2.466638,0.037309,0.008791,-0.100698,-141.972923,1.151388,0.082185,0.132422,0.411480,10.67088,0.048397,9.586974,22.057783,10.668437,4.288498e+05,1.747401,0.848225,4.681957,0.089612,0.000000,2025-08-31
001395.XSHE,-2.184201,0.199910,0.010011,2.351318,874.814695,0.938933,0.080335,0.158675,0.653743,16.42451,0.017582,0.151235,22.339083,26.352683,8.386478e+05,2.082720,0.893635,0.109104,0.154546,0.000000,2025-08-31
001256.XSHE,-2.613876,0.031183,0.010616,-1.838565,-27.970384,0.684368,0.049255,0.154242,0.612186,8.31860,0.022485,0.448993,21.911137,8.388313,1.275893e+06,1.208164,0.889375,0.382140,0.070726,0.000000,2025-08-31


In [7]:

import pandas as pd
import numpy as np
from jqdata import *  # 假设你用的是聚宽API
import pandas as pd
import numpy as np
from jqlib.technical_analysis import *
from jqfactor import get_factor_values
from jqfactor import winsorize_med
from jqfactor import standardlize
from jqfactor import neutralize


def filter_st_stock(stock_list, date):
    """
    :param stock_list: 股票代码列表
    :param date: 日期字符串，如 '2025-06-26'
    """
    is_st_series = get_extras('is_st', stock_list, start_date=date, end_date=date).iloc[0]
    return [stock for stock in stock_list if not is_st_series[stock]]

    

#2-3 过滤科创北交股票
def filter_kcbj_stock(stock_list):
    for stock in stock_list[:]:
        if stock[0] == '4' or stock[0] == '8' or stock[:2] == '68':
            stock_list.remove(stock)
    return stock_list

#2-6 过滤次新股
def filter_new_stock(yesterday,stock_list):
    if isinstance(yesterday, str):
        yesterday = datetime.datetime.strptime(yesterday, "%Y-%m-%d").date()
    
    return [stock for stock in stock_list 
            if not (yesterday - get_security_info(stock).start_date < datetime.timedelta(days=375))]


def filter_paused_stock(stock_list, date):
    """
    过滤停牌的股票（适用于回测）
    
    :param stock_list: 股票代码列表
    :param date: 当前判断的日期，格式为 datetime.date 或字符串 'YYYY-MM-DD'
    :return: 未停牌的股票列表
    """
    valid_stocks = []
    for stock in stock_list:
        df = get_price(stock, start_date=date, end_date=date, frequency='daily', fields=['volume'])
        if not df.empty and df['volume'].iloc[0] > 0:
            valid_stocks.append(stock)
    return valid_stocks


def filter_limitup_stock(stock_list):
    close_df = history(1, unit='1m', field='close', security_list=stock_list)
    high_limit_df = history(1, unit='1m', field='high_limit', security_list=stock_list)

    result = []
    for stock in stock_list:
        if close_df[stock][-1] < high_limit_df[stock][-1]:
            result.append(stock)
    return result


def filter_limitdown_stock(stock_list):
    close_df = history(1, unit='1m', field='close', security_list=stock_list)
    low_limit_df = history(1, unit='1m', field='low_limit', security_list=stock_list)

    result = []
    for stock in stock_list:
        if close_df[stock][-1] > low_limit_df[stock][-1]:
            result.append(stock)
    return result





print("开始运行选股打分程序")


# 获取当前日期
y = datetime.datetime.now() - datetime.timedelta(days=1)
y -= datetime.timedelta(days=max(0, y.weekday() - 4))

today = y.strftime('%Y-%m-%d')

print(f"今日日期: {today}")

# stockList=get_index_stocks('399101.XSHE',today)
# initial_list = [stock for stock in stockList if not stock.startswith(('68', '4', '8'))]
        
# # 获取初始股票列表
initial_list = get_all_securities('stock', today).index.tolist()
print(f"初始股票数量: {len(initial_list)}")

# 过滤次新股
initial_list = filter_new_stock(today, initial_list)
print(f"过滤次新股后股票数量: {len(initial_list)}")

# 过滤科创板北交所股票
initial_list = filter_kcbj_stock(initial_list)
print(f"过滤科创板北交所后股票数量: {len(initial_list)}")

# 过滤ST股票
initial_list = filter_st_stock(initial_list,today)
print(f"过滤ST股票后股票数量: {len(initial_list)}")

total_scores = pd.Series(0, index=initial_list, dtype=float)
valid_stocks_set = set(initial_list)


import pandas as pd

stock_num =1

final_list=[]
for model_idx, (factors, coefs,incp) in enumerate(factor_list):
    print(f"模型{model_idx}，因子: {factors}")

    factor_values = get_factor_values(initial_list, factors, end_date=today, count=1)

    df = pd.DataFrame(index=initial_list, columns=factors)
    for f in factors:
        df[f] = list(factor_values[f].T.iloc[:, 0])
    df = df.dropna()
    
    df['total_score'] = 0
    for f, c in zip(factors, coefs):
        df['total_score'] += c * df[f]

#     df['total_score'] = incp  # 加上截距项
#     # 标准化后再加权打分
#     for f, c in zip(factors, coefs):
#         df['total_score'] += c * ((df[f] - df[f].mean()) / df[f].std())    
        
    df = df.sort_values(by='total_score', ascending=False)
    top_k = max(2, int(0.10 * len(df)))
    complex_factor_list = list(df.index)[:top_k]

    q = query(valuation.code, valuation.circulating_market_cap, indicator.eps)\
        .filter(valuation.code.in_(complex_factor_list))\
        .order_by(valuation.circulating_market_cap.asc())
    basic_df = get_fundamentals(q)
    basic_df = basic_df[basic_df['eps'] > 0]

    lst = list(basic_df.code)
    lst = filter_paused_stock(lst,today)
    lst = filter_limitup_stock(lst)
    lst = filter_limitdown_stock(lst)
    lst = lst[:min(stock_num, len(lst))]
    
    for stock in lst:
        if stock not in final_list:
            final_list.append(stock)
            
        

print("\n📋 本模型最终选中股票：")
for stock in final_list:
    stock_name = get_security_info(stock).display_name 
    print(f"{stock} - {stock_name}")


开始运行选股打分程序
今日日期: 2025-11-10
初始股票数量: 5167
过滤次新股后股票数量: 5080
过滤科创板北交所后股票数量: 4503
过滤ST股票后股票数量: 4331
模型0，因子: ['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap']
模型1，因子: ['size', 'roe_ttm', 'current_asset_turnover_rate']
模型2，因子: ['VOL10', 'single_day_VPT_12']
模型3，因子: ['adjusted_profit_to_total_profit']
模型4，因子: ['super_quick_ratio', 'cube_of_size', 'cfo_to_ev']
模型5，因子: ['cash_to_current_liability', 'operating_tax_to_operating_revenue_ratio_ttm', 'Price3M']
模型6，因子: ['liquidity', 'roa_ttm']
模型7，因子: ['VSTD10', 'ROC60']

📋 本模型最终选中股票：
301429.XSHE - 森泰股份
001260.XSHE - 坤泰股份
301053.XSHE - 远信工业
301520.XSHE - 万邦医药
603280.XSHG - 南方路机
600883.XSHG - 博闻科技


In [8]:
# import requests
# import json

# # ---------- 假设这是你从字符串提取出来的股票代码 ----------
# test_stocks = final_list
# # [
# #     "001231.XSHE - 农心科技",
# #     "001260.XSHE - 坤泰股份",
# #     "301088.XSHE - 戎美股份",
# #     "001326.XSHE - 联域股份",
# #     "301567.XSHE - 贝隆精密",
# #     "301081.XSHE - 严牌股份"
# # ]

# # 提取纯股票代码
# codes = [s.split(".")[0] for s in test_stocks]

# print(codes)  # 预期 ["001231", "001260", "301088", "001326", "301567", "301081"]

# # 构造 FC 事件参数
# event = {"codes": codes}

# # ---------- 调用 FC HTTP Trigger ----------
# fc_url = "http://bestok.cn/gegu"
# headers = {"Content-Type": "application/json"}

# response = requests.post(fc_url, headers=headers, data=json.dumps(event))

# # ---------- 输出 FC 返回结果 ----------
# print(response.status_code)
# print(response.json())

# import pandas as pd

# # 假设这是你的返回结果
# res = response.json()

# # 提取 data 并转成 DataFrame
# df = pd.DataFrame(res.get("data", []))

# # 打印结果
# df



In [9]:
# import pandas as pd
# import numpy as np
# import datetime

# print("开始运行选股打分程序")

# # 昨天，遇周末直接跳到上周五
# y = datetime.datetime.now() - datetime.timedelta(days=1)
# y -= datetime.timedelta(days=max(0, y.weekday() - 4))

# # 生成2001年到2005年9月3日的交易日列表
# start_date = '2025-08-01'
# end_date = y.strftime('%Y-%m-%d')


# trading_days = get_trade_days(start_date, end_date)

# # 用于存储所有选中的股票及其日期
# all_selected_stocks = []

# # 遍历每个交易日
# for today in trading_days:
#     print(f"处理日期: {today}")
    
#     # 获取初始股票列表
#     try:
#         initial_list = get_all_securities('stock', today).index.tolist()
#         print(f"初始股票数量: {len(initial_list)}")
        
#         # 过滤次新股
#         initial_list = filter_new_stock(today, initial_list)
# #         print(f"过滤次新股后股票数量: {len(initial_list)}")
        
#         # 过滤科创板北交所股票（注意：2001-2005年还没有科创板和北交所）
#         initial_list = filter_kcbj_stock(initial_list)
# #         print(f"过滤科创板北交所后股票数量: {len(initial_list)}")
        
#         # 过滤ST股票
#         initial_list = filter_st_stock(initial_list, today)
# #         print(f"过滤ST股票后股票数量: {len(initial_list)}")
        
#         total_scores = pd.Series(0, index=initial_list, dtype=float)
#         valid_stocks_set = set(initial_list)
        
#         stock_num = 1
#         final_list = []
        
#         for model_idx, (factors, coefs, incp) in enumerate(factor_list):
# #             print(f"模型{model_idx}，因子: {factors}")

#             factor_values = get_factor_values(initial_list, factors, end_date=today, count=1)

#             df = pd.DataFrame(index=initial_list, columns=factors)
#             for f in factors:
#                 df[f] = list(factor_values[f].T.iloc[:, 0])
#             df = df.dropna()
            
#             df['total_score'] = 0
#             for f, c in zip(factors, coefs):
#                 df['total_score'] += c * df[f]
                
#             df = df.sort_values(by='total_score', ascending=False)
#             top_k = max(2, int(0.10 * len(df)))
#             complex_factor_list = list(df.index)[:top_k]

#             q = query(valuation.code, valuation.circulating_market_cap, indicator.eps)\
#                 .filter(valuation.code.in_(complex_factor_list))\
#                 .order_by(valuation.circulating_market_cap.asc())
#             basic_df = get_fundamentals(q)
#             basic_df = basic_df[basic_df['eps'] > 0]

#             lst = list(basic_df.code)
#             lst = filter_paused_stock(lst, today)
#             lst = filter_limitup_stock(lst)
#             lst = filter_limitdown_stock(lst)
#             lst = lst[:min(stock_num, len(lst))]
            
#             for stock in lst:
#                 if stock not in final_list:
#                     final_list.append(stock)
        
#         # 将选中的股票添加到总列表中
#         for stock in final_list:
#             all_selected_stocks.append({
#                 'code': stock,
#                 'date': today.strftime('%Y-%m-%d'),
#                 'weight': 0.1
#             })
            
#     except Exception as e:
#         print(f"处理日期 {today} 时出错: {str(e)}")
#         continue

# # 将结果转换为DataFrame并保存为CSV文件
# result_df = pd.DataFrame(all_selected_stocks)
# result_df = result_df[['code', 'date', 'weight']]  # 确保列顺序正确

# print("\n📋 选股结果统计：")
# print(f"总共选中 {len(result_df)} 条记录")
# print(f"涉及 {result_df['code'].nunique()} 只不同的股票")
# print(f"时间跨度：{result_df['date'].min()} 到 {result_df['date'].max()}")

# # 保存结果
# result_df.to_csv('selected_stocks_2001_2005.csv', index=False, header=False)
# print("\n结果已保存到 selected_stocks_2005_2005.csv 文件")

# # 显示前20条记录作为示例
# print("\n前20条记录示例：")
# print(result_df.head(20).to_string(index=False))

In [10]:
import pandas as pd
import numpy as np

# jqfactors_list 需提前定义好
# jqfactors_list = [...]
# 
X_test = pd.read_csv(test_filename)

# 替换 inf/-inf 为 NaN
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

df_filled_test = X_test.copy()
columns_to_fill = [col for col in jqfactors_list if col in X_test.columns]

for column in columns_to_fill:
    # 先按日期分组填充
    df_filled_test[column] = df_filled_test.groupby('date')[column].transform(
        lambda x: x.fillna(x.median())
    )
    # 如果还有 NaN，用全局中位数填充
    if df_filled_test[column].isna().any():
        global_median = df_filled_test[column].median()
        df_filled_test[column].fillna(global_median, inplace=True)

# 去除仍包含 NaN 的行（如果还有）
X_test_cleaned = df_filled_test.dropna().reset_index(drop=True)

X_test = X_test_cleaned

X_test.rename(columns={'Unnamed: 0': 'id'}, inplace=True)

print(f"测试集样本数: {len(X_test)}")

y_test =X_test['pchg']



测试集样本数: 66380


In [11]:



from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1. 定义获取股票名称函数
def get_stock_name(stock_code):
    return get_security_info(stock_code).display_name


def get_top_scoring_stocks(X_test, factor_list, model_idx):
    """根据因子模型计算得分并返回 top 10% 股票"""
    factors, coefs, intercept = factor_list
    X_part = X_test[factors].copy()
    score = intercept + np.dot(X_part.values, np.array(coefs))

    temp_df = pd.DataFrame({
        'code': X_test['id'].values,
        'date': X_test['date'].values,
        'score': score
    }).dropna()

    temp_df_sorted = temp_df.sort_values(by='score', ascending=False)
    top_k = max(2, int(0.10 * len(temp_df_sorted)))
    return temp_df_sorted.iloc[:top_k]


def filter_by_fundamental_and_risk(stock_list, today, stock_num):
    """根据 EPS 和流通市值、停牌、涨跌停过滤股票"""
    if not stock_list:
        return []

    q = query(valuation.code, valuation.circulating_market_cap, indicator.eps)\
        .filter(valuation.code.in_(stock_list))\
        .order_by(valuation.circulating_market_cap.asc())

    basic_df = get_fundamentals(q)
    basic_df = basic_df[basic_df['eps'] > 0]

    filtered = list(basic_df.code)
    filtered = filter_paused_stock(filtered, today)
    filtered = filter_limitup_stock(filtered)
    filtered = filter_limitdown_stock(filtered)

    return filtered[:min(stock_num, len(filtered))], basic_df


def process_model_result(model_idx, factor_list, X_test, y_test, today, stock_num):
    top_rows = get_top_scoring_stocks(X_test, factor_list, model_idx)
    complex_factor_list = list(top_rows['code'])

    valid_stocks, basic_df = filter_by_fundamental_and_risk(complex_factor_list, today, stock_num)

    selected = []
    records = []

    for _, row in top_rows.iterrows():
        stock, date, score_val = row['code'], row['date'], row['score']
        if stock not in valid_stocks:
            continue

        name = get_stock_name(stock)
        stock_mask = (X_test['id'] == stock) & (X_test['date'] == date)
        ret = y_test[stock_mask].iloc[-1] if not y_test[stock_mask].empty else np.nan
        cap = basic_df.loc[basic_df.code == stock, 'circulating_market_cap'].values[0] \
              if stock in list(basic_df.code) else np.nan

        selected.append(stock)
        records.append({
            'code': stock,
            'name': name,
            'date': date,
            'return': ret,
            'score': score_val,
            'circulating_market_cap': cap
        })
    # 打印 Top 股票信息（可选）
    print(f"\n📈 模型 {model_idx + 1}: 使用因子 {factor_list[0]}")
    print("🧾 Top 股票详情（含累计回报）:")

    for rec in records:
        code, name, date, ret, score_val = rec['code'], rec['name'], rec['date'], rec['return'], rec['score']

        # 获取该股票的所有历史日期和回报
        stock_mask = X_test['id'] == code
        stock_df = pd.DataFrame({
            'date': X_test.loc[stock_mask, 'date'].values,
            'return': y_test[stock_mask].values
        }).sort_values(by='date').reset_index(drop=True)

        if stock_df.empty or date not in stock_df['date'].values:
            print(f"⚠️ {name} ({code}) 在 date={date} 未找到历史数据，跳过打印。")
            continue

        # 找到当前日期对应的 index
        idx = stock_df[stock_df['date'] == date].index[0]

        # 向前最多5日累计回报
        start_fwd = max(idx - 4, 0)
        forward_df = stock_df.iloc[start_fwd:idx + 1]
        forward_return = (forward_df['return'] + 1).prod() - 1
        forward_series = forward_df['return'].tolist()

        # 向后最多100日累计回报
        end_bwd = min(idx + 100, len(stock_df))
        backward_df = stock_df.iloc[idx:end_bwd]
        backward_return = (backward_df['return'] + 1).prod() - 1
        backward_series = backward_df['return'].tolist()

        # 打印结果
        print(f"\n{name} ({code}) - 当前日期: {date}，当前回报: {ret:.2%}，因子得分: {score_val:.4f}")
        print(f"📅 向前5日累计回报: {forward_return:.2%}，序列: {[f'{x:.2%}' for x in forward_series]}")
        print(f"📅 向后100日累计回报: {backward_return:.2%}，序列: {[f'{x:.2%}' for x in backward_series]}")

    return selected, records



# === 主调用逻辑 ===
final_list = []
top_stock_records = []

for i, factor_model in enumerate(factor_list):
    selected_stocks, records = process_model_result(i, factor_model, X_test, y_test, today, stock_num)
    final_list.extend(selected_stocks)
    top_stock_records.extend(records)

top_stock_df = pd.DataFrame(top_stock_records)



# 假设你的原始 DataFrame 名为 df



# 显示所有列
pd.set_option('display.max_columns', None)

# 显示所有行（慎用，数据多时可能刷屏）
pd.set_option('display.max_rows', None)

# 禁止列内容缩略（比如长字符串）
pd.set_option('display.max_colwidth', 1000)

# 防止 DataFrame 在输出时被换行折叠
pd.set_option('display.expand_frame_repr', False)

# # === 如果你想只看某一天的结果，比如2025-06-27 ===
# df_day = top_stock_df[top_stock_df['date'] == '2025-06-27']
# df_sorted = df_day.sort_values(by='circulating_market_cap')

# # 输出结果
# print(df_sorted)

# 保留每只股票最新日期的记录
df_latest = top_stock_df.sort_values(by='date', ascending=False).drop_duplicates(subset='code', keep='first')

# 按 circulating_market_cap 从小到大排序
df_sorted = df_latest.sort_values(by='circulating_market_cap')


print(df_sorted)



📈 模型 1: 使用因子 ['sales_to_price_ratio', 'share_turnover_monthly', 'natural_log_of_market_cap']
🧾 Top 股票详情（含累计回报）:

楚环科技 (001336.XSHE) - 当前日期: 2025-10-17，当前回报: 3.55%，因子得分: 0.0028
📅 向前5日累计回报: -0.39%，序列: ['-0.35%', '-0.39%', '-2.44%', '-0.67%', '3.55%']
📅 向后100日累计回报: 17.26%，序列: ['3.55%', '2.78%', '1.48%', '0.17%', '1.95%', '-0.29%', '1.23%', '-0.61%', '0.57%', '1.94%', '0.83%', '0.63%', '1.13%', '1.16%', '0.11%', '-0.53%']

楚环科技 (001336.XSHE) - 当前日期: 2025-10-16，当前回报: -0.67%，因子得分: 0.0028
📅 向前5日累计回报: -4.14%，序列: ['-0.34%', '-0.35%', '-0.39%', '-2.44%', '-0.67%']
📅 向后100日累计回报: 16.47%，序列: ['-0.67%', '3.55%', '2.78%', '1.48%', '0.17%', '1.95%', '-0.29%', '1.23%', '-0.61%', '0.57%', '1.94%', '0.83%', '0.63%', '1.13%', '1.16%', '0.11%', '-0.53%']

📈 模型 2: 使用因子 ['size', 'roe_ttm', 'current_asset_turnover_rate']
🧾 Top 股票详情（含累计回报）:

坤泰股份 (001260.XSHE) - 当前日期: 2025-09-22，当前回报: -0.26%，因子得分: 0.0026
📅 向前5日累计回报: -4.18%，序列: ['0.10%', '-1.49%', '-1.41%', '-1.18%', '-0.26%']
📅 向后100日累计回报: 8.50%，序列: ['-0.26%'


坤泰股份 (001260.XSHE) - 当前日期: 2025-09-18，当前回报: -1.41%，因子得分: 0.0026
📅 向前5日累计回报: -1.91%，序列: ['-0.50%', '1.41%', '0.10%', '-1.49%', '-1.41%']
📅 向后100日累计回报: 5.71%，序列: ['-1.41%', '-1.18%', '-0.26%', '2.60%', '-0.51%', '-0.15%', '1.48%', '0.25%', '-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%', '1.98%', '0.82%', '0.24%', '0.82%', '0.90%', '-0.61%', '-0.66%']

坤泰股份 (001260.XSHE) - 当前日期: 2025-10-29，当前回报: -0.59%，因子得分: 0.0026
📅 向前5日累计回报: -0.98%，序列: ['1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%']
📅 向后100日累计回报: 2.90%，序列: ['-0.59%', '1.98%', '0.82%', '0.24%', '0.82%', '0.90%', '-0.61%', '-0.66%']

坤泰股份 (001260.XSHE) - 当前日期: 2025-09-26，当前回报: 1.48%，因子得分: 0.0026
📅 向前5日累计回报: 3.16%，序列: ['-0.26%', '2.60%', '-0.51%', '-0.15%', '1.48%']
📅 向后100日累计回报: 6.73%，序列: ['1.48%', '0.25%', '-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%',


联域股份 (001326.XSHE) - 当前日期: 2025-10-21，当前回报: -1.27%，因子得分: 0.0013
📅 向前5日累计回报: -6.87%，序列: ['-1.42%', '-1.84%', '-1.13%', '-1.41%', '-1.27%']
📅 向后100日累计回报: -7.95%，序列: ['-1.27%', '-0.28%', '0.52%', '1.21%', '-2.31%', '-2.04%', '-2.41%', '0.44%', '-1.27%', '1.05%', '-0.84%', '2.32%', '-1.23%', '-2.00%']

联域股份 (001326.XSHE) - 当前日期: 2025-08-31，当前回报: 0.00%，因子得分: 0.0013
📅 向前5日累计回报: 0.00%，序列: ['0.00%']
📅 向后100日累计回报: 23.40%，序列: ['0.00%', '-2.41%', '-1.16%', '-2.01%', '2.80%', '1.46%', '-0.77%', '0.38%', '1.92%', '-0.68%', '-0.34%', '3.20%', '4.82%', '4.53%', '3.14%', '1.73%', '1.56%', '8.29%', '-5.56%', '0.32%', '0.86%', '0.83%', '10.01%', '-3.50%', '-0.99%', '1.70%', '6.76%', '-1.42%', '-1.84%', '-1.13%', '-1.41%', '-1.27%', '-0.28%', '0.52%', '1.21%', '-2.31%', '-2.04%', '-2.41%', '0.44%', '-1.27%', '1.05%', '-0.84%', '2.32%', '-1.23%', '-2.00%']

联域股份 (001326.XSHE) - 当前日期: 2025-10-13，当前回报: 1.70%，因子得分: 0.0013
📅 向前5日累计回报: 7.77%，序列: ['0.83%', '10.01%', '-3.50%', '-0.99%', '1.70%']
📅 向后100日累计回报: -


坤泰股份 (001260.XSHE) - 当前日期: 2025-09-10，当前回报: 1.06%，因子得分: 0.0033
📅 向前5日累计回报: 3.82%，序列: ['1.29%', '1.43%', '0.55%', '-0.55%', '1.06%']
📅 向后100日累计回报: 5.28%，序列: ['1.06%', '-0.95%', '-0.50%', '1.41%', '0.10%', '-1.49%', '-1.41%', '-1.18%', '-0.26%', '2.60%', '-0.51%', '-0.15%', '1.48%', '0.25%', '-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%', '1.98%', '0.82%', '0.24%', '0.82%', '0.90%', '-0.61%', '-0.66%']

坤泰股份 (001260.XSHE) - 当前日期: 2025-09-04，当前回报: 1.29%，因子得分: 0.0033
📅 向前5日累计回报: -7.68%，序列: ['0.00%', '-6.26%', '-3.02%', '0.26%', '1.29%']
📅 向后100日累计回报: 8.17%，序列: ['1.29%', '1.43%', '0.55%', '-0.55%', '1.06%', '-0.95%', '-0.50%', '1.41%', '0.10%', '-1.49%', '-1.41%', '-1.18%', '-0.26%', '2.60%', '-0.51%', '-0.15%', '1.48%', '0.25%', '-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%', '1.98%', '


坤泰股份 (001260.XSHE) - 当前日期: 2025-10-09，当前回报: 0.00%，因子得分: 0.0032
📅 向前5日累计回报: 0.81%，序列: ['-0.15%', '1.48%', '0.25%', '-0.75%', '0.00%']
📅 向后100日累计回报: 5.71%，序列: ['0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%', '1.98%', '0.82%', '0.24%', '0.82%', '0.90%', '-0.61%', '-0.66%']

坤泰股份 (001260.XSHE) - 当前日期: 2025-09-29，当前回报: 0.25%，因子得分: 0.0031
📅 向前5日累计回报: 3.69%，序列: ['2.60%', '-0.51%', '-0.15%', '1.48%', '0.25%']
📅 向后100日累计回报: 5.18%，序列: ['0.25%', '-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95%', '0.74%', '1.91%', '-0.72%', '0.39%', '-1.93%', '-0.59%', '1.98%', '0.82%', '0.24%', '0.82%', '0.90%', '-0.61%', '-0.66%']

坤泰股份 (001260.XSHE) - 当前日期: 2025-09-30，当前回报: -0.75%，因子得分: 0.0031
📅 向前5日累计回报: 0.30%，序列: ['-0.51%', '-0.15%', '1.48%', '0.25%', '-0.75%']
📅 向后100日累计回报: 4.91%，序列: ['-0.75%', '0.00%', '-0.30%', '0.10%', '0.46%', '-1.06%', '-0.20%', '1.33%', '1.11%', '0.95


博闻科技 (600883.XSHG) - 当前日期: 2025-10-15，当前回报: -0.85%，因子得分: 0.0039
📅 向前5日累计回报: 0.99%，序列: ['1.36%', '-0.49%', '0.86%', '0.12%', '-0.85%']
📅 向后100日累计回报: 6.68%，序列: ['-0.85%', '0.00%', '1.10%', '1.33%', '0.48%', '0.36%', '0.00%', '0.36%', '0.47%', '-1.41%', '-0.84%', '1.68%', '0.83%', '0.82%', '1.40%', '-0.11%', '0.11%', '0.80%']

博闻科技 (600883.XSHG) - 当前日期: 2025-09-09，当前回报: 0.36%，因子得分: 0.0039
📅 向前5日累计回报: 5.77%，序列: ['1.67%', '0.63%', '2.76%', '0.24%', '0.36%']
📅 向后100日累计回报: 6.81%，序列: ['0.36%', '0.24%', '-0.48%', '0.73%', '0.24%', '-0.72%', '-1.82%', '-0.49%', '-1.99%', '-1.90%', '1.94%', '-0.89%', '0.89%', '1.65%', '-0.25%', '0.87%', '1.36%', '-0.49%', '0.86%', '0.12%', '-0.85%', '0.00%', '1.10%', '1.33%', '0.48%', '0.36%', '0.00%', '0.36%', '0.47%', '-1.41%', '-0.84%', '1.68%', '0.83%', '0.82%', '1.40%', '-0.11%', '0.11%', '0.80%']

博闻科技 (600883.XSHG) - 当前日期: 2025-10-14，当前回报: 0.12%，因子得分: 0.0039
📅 向前5日累计回报: 2.75%，序列: ['0.87%', '1.36%', '-0.49%', '0.86%', '0.12%']
📅 向后100日累计回报: 6.81%，序列: ['0.1